In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when

# Impor library MLlib
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import (
     RandomForestClassifier, GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import pandas as pd

# Hentikan SparkSession jika ada yang aktif
try:
    spark.stop()
except:
    pass

# Buat SparkSession baru
spark = SparkSession.builder \
    .appName("ChurnModeling") \
    .config("spark.driver.memory", "12g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

print("SparkSession dan library MLlib siap.")

SparkSession dan library MLlib siap.


In [2]:
data_path = "data_final/master_feature_table_final.parquet"
df = spark.read.parquet(data_path)
df.printSchema()

root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- city: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_plan_days: long (nullable = true)
 |-- total_amount_paid: double (nullable = true)
 |-- avg_amount_paid: double (nullable = true)
 |-- count_auto_renew: long (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- most_frequent_payment_method: integer (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = true)
 |-- active_days_last_30d: long (nullable = true)
 |-- total_secs_last_90d: double (nullable = true)
 |-- active_days_last_90d: long (nullable = true)
 |-- percent_complete_last_30d: double (nullable = true)
 |-- lifetime_active_days: long (nullable = true)
 |-- lifetime_unq_songs: long (nullable = true)



In [3]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd


# --- 2. PREPARE DATA (pilih cols) ---
    
cols_to_keep = [
    "msno",
    "is_churn",
    # "city",
    # "age_group",
    # "registered_via",
    "total_transactions",
    # "total_plan_days",
    # "total_amount_paid",
    "avg_amount_paid",
    "count_auto_renew",
    "count_cancel",
    "most_frequent_payment_method",
    "days_since_last_activity",
    "total_secs_last_30d",
    "active_days_last_30d",
    # "total_secs_last_90d",
    # "active_days_last_90d",
    # "percent_complete_last_30d",
    "lifetime_active_days",
    # "lifetime_unq_songs"
]




# Ambil cmn kolom yang diminta dari master table
df_selected = df.select(*cols_to_keep)


print("Skema Final Data Latih:")
df_selected.printSchema()
print(df_selected.count())

Skema Final Data Latih:
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- avg_amount_paid: double (nullable = true)
 |-- count_auto_renew: long (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- most_frequent_payment_method: integer (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = true)
 |-- active_days_last_30d: long (nullable = true)
 |-- lifetime_active_days: long (nullable = true)

1082190


In [ ]:
# from pyspark.sql import functions as F
# from pyspark.ml import Pipeline
# from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
# from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# import pandas as pd


# # --- 2. PREPARE DATA PER BAGIAN ---

# # --- BAGIAN B: TRANSAKSI (Agregasi per User) ---
# print("Mengagregasi Transaksi...")
# trans_agg = transactions_df.groupBy("msno").agg(
#     F.count("payment_method_id").alias("total_transactions"),
#     # F.sum("payment_plan_days").alias("total_plan_days"),
#     # F.sum("actual_amount_paid").alias("total_amount_paid"),
#     F.avg("actual_amount_paid").alias("avg_amount_paid"),
#     F.sum("is_auto_renew").alias("count_auto_renew"),
#     F.sum("is_cancel").alias("count_cancel"),
#     F.mode("payment_method_id").alias("most_frequent_payment_method")
# )

# # --- BAGIAN C: LOGS (Seleksi Kolom pakai fitur lama di proposal) ---
# print("Menyeleksi Kolom Logs...")
# cols_to_keep = [
#     "msno",
#     "days_since_last_activity",
#     "total_secs_last_30d",
#     "active_days_last_30d",
#     # "total_secs_last_90d",
#     # "active_days_last_90d",
#     # "percent_complete_last_30d",
#     "lifetime_active_days",
#     # "lifetime_unq_songs"
# ]


# # Ambil cmn kolom yang diminta dari master table
# logs_selected = logs_df_new.select(*cols_to_keep)

# membercols_to_keep= [
#     "msno",
#     "is_churn",
#     # "registered_via"
#     # "age_group",
#     # "city"
# ]

# members_selected = members_df.select(*membercols_to_keep)
# # --- 3. JOIN SEMUA TABEL ---
# print("Menggabungkan Tabel (Members + Transaksi + Logs Baru)...")

# # Left Join agar kita tetap patokannya user Remaja
# final_df = members_selected.join(trans_agg, "msno", "left") \
#                               .join(logs_selected, "msno", "left")

# # --- 4. CLEANING & FILLNA ---
# # Isi 0 untuk fitur numerik yang kosong (karena user tidak punya transaksi/logs)
# # kita harus mengisi 0 untuk kolom logs baru juga
# # final_df_model = final_df.fillna(0, subset=[
# #     "total_transactions", "total_plan_days", "total_amount_paid", 
# #     "avg_amount_paid", "count_auto_renew", "count_cancel",
# #     "most_frequent_payment_method",
# #     "days_since_last_activity", "total_secs_last_30d", "active_days_last_30d", "activity_ratio_secs",
# #     "percent_complete_last_30d", "lifetime_active_days", "lifetime_unq_songs"
# # ])

# final_df_model = final_df.fillna(0, subset=[
#     "total_transactions", 
#     # "total_plan_days", 
#     # "total_amount_paid", 
#     "avg_amount_paid", "count_auto_renew", "count_cancel",
#     # "percent_complete_last_30d",
#     "most_frequent_payment_method", 
#     "days_since_last_activity", "total_secs_last_30d", "active_days_last_30d", 
#     "lifetime_active_days", 
#     # "lifetime_unq_songs"
# ])

# # final_df_model = final_df.fillna(0, subset=[
# #     "total_transactions", "total_plan_days", 
# #     "avg_amount_paid", "count_auto_renew", "count_cancel",
# #     "most_frequent_payment_method",
# #     "days_since_last_activity", "total_secs_last_30d", "active_days_last_30d", 
# #     "lifetime_active_days"
# # ])
# # Isi Unknown untuk kategorikal


# print("Skema Final Data Latih:")
# final_df_model.printSchema()
# print(final_df_model.count())

Mengagregasi Transaksi...
Menyeleksi Kolom Logs...
Menggabungkan Tabel (Members + Transaksi + Logs Baru)...
Skema Final Data Latih:
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- avg_amount_paid: double (nullable = false)
 |-- count_auto_renew: long (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- most_frequent_payment_method: integer (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = false)
 |-- active_days_last_30d: long (nullable = true)
 |-- lifetime_active_days: long (nullable = true)

1082190


### Model Pipeline

In [ ]:
# --- 5. PIPELINE STAGES ---
label_col = "is_churn"
# Hanya 'age_group' sebagai kategori
# categorical_cols = ["registered_via"] 

# Semua kolom lain selain 'is_churn', 'msno', dan kategorikal adalah numerik
numerical_cols = [
    col for col in df_selected.columns 
    if col not in [label_col, "msno"]
]

# print(f"\nFitur Kategorikal: {categorical_cols}")
print(f"Fitur Numerik ({len(numerical_cols)}): {numerical_cols}")

stages = []

# # Tahap 1: StringIndexer (Hanya untuk data kategori)
# for col in categorical_cols:
#     indexer = StringIndexer(
#         inputCol=col, 
#         outputCol=f"{col}_idx", 
#         handleInvalid="keep"
#     )
#     stages += [indexer]

# # Tahap 2: OneHotEncoder
# encoder = OneHotEncoder(
#     inputCols=[f"{col}_idx" for col in categorical_cols],
#     outputCols=[f"{col}_vec" for col in categorical_cols]
# )
# stages += [encoder]

# Tahap 3: VectorAssembler (Hanya untuk fitur NUMERIK)
assembler_num = VectorAssembler(
    inputCols=numerical_cols, 
    outputCol="numerical_features"
)
stages += [assembler_num]

# Tahap 4: StandardScaler (Untuk fitur numerik)
scaler = StandardScaler(
    inputCol="numerical_features", 
    outputCol="scaled_numerical_features"
)
stages += [scaler]

# Tahap 5: VectorAssembler Final (Menggabungkan SEMUA fitur)
input_vecs =  ["scaled_numerical_features"]
final_assembler = VectorAssembler(
    inputCols=input_vecs,
    outputCol="features"
)
stages += [final_assembler]

Fitur Numerik (9): ['total_transactions', 'avg_amount_paid', 'count_auto_renew', 'count_cancel', 'most_frequent_payment_method', 'days_since_last_activity', 'total_secs_last_30d', 'active_days_last_30d', 'lifetime_active_days']


### Tanpa Oversampling/Undersampling

In [7]:
print("\nMembagi Data Latih (80%) & Uji (20%)...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

print("Melakukan Oversampling...")
print(f"data_train:{train_data.count()}")
print(f"data_test: {test_data.count()}")


Membagi Data Latih (80%) & Uji (20%)...
Melakukan Oversampling...
data_train:865693
data_test: 216497


### Oversampling

In [17]:
# --- 6. SPLIT & OVERSAMPLING (15%) ---

print("Membagi Data Latih (80%) & Uji (20%)...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

print("Melakukan Oversampling...")
print(f"Jumlah data train: {train_data.count()}")
print(f"Jumlah data test: {test_data.count()}")


major_df = train_data.filter(F.col(label_col) == 0)
minor_df = train_data.filter(F.col(label_col) == 1)


count_major = major_df.count()
count_minor = minor_df.count()

# Tentukan rasio tambahan (12.5%)
ratio_tambahan = (0.125 * train_data.count() - count_minor)/count_minor
print(f"ratio tambahan: {ratio_tambahan} dari data minoritas")

# Ambil sampel tambahan (Hanya mengambil 12.5% dari keseluruhan data ekstra)
df_minority_extra = minor_df.sample(
    withReplacement=True, 
    fraction=ratio_tambahan, 
    seed=42
)
train_data_balanced = major_df.unionAll(df_minority_extra).unionAll(minor_df)

minor_union = train_data_balanced.filter(F.col(label_col) == 1)

print(f"\nData Latih Awal: {train_data.count()}")
print(f"Data Latih Baru: {train_data_balanced.count()}, bertambah {train_data_balanced.count() - train_data.count()}")

print(f"\nJumlah data major lama (not churn) {count_major}")
print(f"Jumlah data minor lama (churn) {count_minor}")
print(f"Jumlah data minor baru (churn): {minor_union.count()}, bertambah sebanyak {minor_union.count()-count_minor} data")


print(f"Persentase Data Mayoritas Baru: {count_major/train_data_balanced.count()*100:.2f}%")
print(f"Persentase Data Minoritas Baru: {minor_union.count()/train_data_balanced.count()*100:.2f}%")

Membagi Data Latih (80%) & Uji (20%)...
Melakukan Oversampling...
Jumlah data train: 865693
Jumlah data test: 216497
ratio tambahan: 0.36718878318108883 dari data minoritas

Data Latih Awal: 865693
Data Latih Baru: 894595, bertambah 28902

Jumlah data major lama (not churn) 786544
Jumlah data minor lama (churn) 79149
Jumlah data minor baru (churn): 108051, bertambah sebanyak 28902 data
Persentase Data Mayoritas Baru: 87.92%
Persentase Data Minoritas Baru: 12.08%


#### Undersampling

In [19]:
# --- 6. SPLIT & Undersampling (15%) ---

print("\nMembagi Data Latih (80%) & Uji (20%)...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

print("Melakukan Oversampling...")
print(f"Jumlah data train: {train_data.count()}")
print(f"Jumlah data test: {test_data.count()}")

major_df = train_data.filter(F.col(label_col) == 0)
minor_df = train_data.filter(F.col(label_col) == 1)


count_major = major_df.count()
count_minor = minor_df.count()

print(f"Jumlah Awal Mayoritas (0): {count_major:,}")
print(f"Jumlah Awal Minoritas (1): {count_minor:,}")

# Tentukan rasio kurangin (15%)
ratio_ambil_major = 1-(0.15 * train_data.count() / count_major)

# Ambil sampel kurangin (Kurangi 15% dr keseluruhan)
df_majority_sample = major_df.sample(
    withReplacement=True, 
    fraction=ratio_ambil_major, 
    seed=42
)

# Gabungkan: Mayoritas Asli + Minoritas Asli + Minoritas Ekstra
train_data_balanced = minor_df.union(df_majority_sample)

minor_union = train_data_balanced.filter(F.col(label_col) == 1)
print(f"count major new: {df_majority_sample.count()}, berkurang sebanyak {count_major-df_majority_sample.count()} data")

print(f"\nData Latih Awal: {train_data.count()}")
print(f"Data Latih Baru: {train_data_balanced.count()}, berkurang {train_data.count() - train_data_balanced.count()}")

print(f"\nJumlah data major lama (not churn): {count_major}")
print(f"Jumlah data major baru (not churn): {df_majority_sample.count()}, berkurang sebanyak {count_major-df_majority_sample.count()} data")
print(f"Jumlah data minor lama (churn): {minor_union.count()}")

print(f"Persentase Data Mayoritas: {df_majority_sample.count()/train_data_balanced.count()*100:.2f}%")
print(f"Persentase Data Minoritas: {minor_union.count()/train_data_balanced.count()*100:.2f}%")


Membagi Data Latih (80%) & Uji (20%)...
Melakukan Oversampling...
Jumlah data train: 865693
Jumlah data test: 216497
Jumlah Awal Mayoritas (0): 786,544
Jumlah Awal Minoritas (1): 79,149
count major new: 655264, berkurang sebanyak 131280 data

Data Latih Awal: 865693
Data Latih Baru: 734413, berkurang 131280

Jumlah data major lama (not churn): 786544
Jumlah data major baru (not churn): 655264, berkurang sebanyak 131280 data
Jumlah data minor lama (churn): 79149
Persentase Data Mayoritas: 89.22%
Persentase Data Minoritas: 10.78%


### Modeling

In [16]:
train_data_balanced.printSchema()

root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- avg_amount_paid: double (nullable = false)
 |-- count_auto_renew: long (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- most_frequent_payment_method: integer (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = false)
 |-- active_days_last_30d: long (nullable = true)
 |-- lifetime_active_days: long (nullable = true)



In [18]:
# --- 7. TRAINING ---

# GBT
print("\n--- Melatih GBT ---")
gbt = GBTClassifier(labelCol=label_col, featuresCol="features", seed=42)
pipeline_gbt = Pipeline(stages=stages + [gbt])
model_gbt = pipeline_gbt.fit(train_data_balanced)
preds_gbt = model_gbt.transform(test_data)

# Random Forest
# print("--- Melatih Random Forest ---")
# rf = RandomForestClassifier(labelCol=label_col, featuresCol="features", seed=42)
# pipeline_rf = Pipeline(stages=stages + [rf])
# model_rf = pipeline_rf.fit(train_data)
# preds_rf = model_rf.transform(test_data)

# --- 8. EVALUASI ---

from pyspark.sql.functions import col

def evaluate_model_detailed(predictions, model_name):
    """
    Menghitung Confusion Matrix, Accuracy, dan 
    Precision/Recall per kelas (Churn & Not Churn)
    menggunakan DataFrame API yang stabil.
    """
    print(f"\n{'='*40}")
    print(f"DETAIL EVALUASI: {model_name}")
    print(f"{'='*40}")
    
    predictions.cache()
    
    # 1. Hitung Komponen Confusion Matrix
    # TP: Prediksi Churn, Aslinya Churn
    tp = predictions.filter((col("prediction") == 1.0) & (col("is_churn") == 1.0)).count()
    # TN: Prediksi Tidak Churn, Aslinya Tidak Churn
    tn = predictions.filter((col("prediction") == 0.0) & (col("is_churn") == 0.0)).count()
    # FP: Prediksi Churn, Tapi Aslinya Tidak (Salah Alarm)
    fp = predictions.filter((col("prediction") == 1.0) & (col("is_churn") == 0.0)).count()
    # FN: Prediksi Tidak Churn, Tapi Aslinya Churn (Lolos)
    fn = predictions.filter((col("prediction") == 0.0) & (col("is_churn") == 1.0)).count()
    
    total = tp + tn + fp + fn
    
    # 2. Hitung Accuracy Global
    accuracy = (tp + tn) / total if total > 0 else 0.0
    
    # 3. Hitung Metrics untuk Kelas 1 (Churn - Positif)
    # Precision 1: Seberapa tepat saat memprediksi Churn?
    prec_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    # Recall 1: Berapa banyak Churn asli yang tertangkap?
    rec_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    # F1 Score 1
    f1_1 = 2 * (prec_1 * rec_1) / (prec_1 + rec_1) if (prec_1 + rec_1) > 0 else 0.0

    # 4. Hitung Metrics untuk Kelas 0 (Not Churn - Negatif)
    # Precision 0: Seberapa tepat saat memprediksi Tidak Churn?
    # (Pembagi adalah Total Prediksi 0 -> TN + FN)
    prec_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    # Recall 0: Berapa banyak User Setia (0) yang benar terdeteksi?
    # (Pembagi adalah Total Asli 0 -> TN + FP)
    rec_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    # F1 Score 0
    f1_0 = 2 * (prec_0 * rec_0) / (prec_0 + rec_0) if (prec_0 + rec_0) > 0 else 0.0
    
    # --- TAMPILKAN HASIL ---
    
    print(f"Total Data Test: {total}")
    print(f"Accuracy       : {accuracy:.2%}")
    
    print(f"\n[ Confusion Matrix ]")
    print(f"      \t Pred 0 \t Pred 1")
    print(f"Aktual 0: {tn}\t\t {fp}")
    print(f"Aktual 1: {fn}\t\t {tp}")

    print("\n--- Metrik per Kelas ---")
    
    # Tabel Manual Rata Kiri
    print(f"{'KELAS':<15} | {'PRECISION':<10} | {'RECALL':<10} | {'F1-SCORE':<10}")
    print("-" * 55)
    print(f"{'0 (Not Churn)':<15} | {prec_0:.2%}     | {rec_0:.2%}     | {f1_0:.2%}")
    print(f"{'1 (Churn)':<15} | {prec_1:.2%}     | {rec_1:.2%}     | {f1_1:.2%}")
    print("-" * 55)
    
    predictions.unpersist()

evaluate_model_detailed(preds_gbt, "GBT Classifier")
# evaluate_model_detailed(preds_rf, "Random Forest")


--- Melatih GBT ---

DETAIL EVALUASI: GBT Classifier
Total Data Test: 216497
Accuracy       : 95.30%

[ Confusion Matrix ]
      	 Pred 0 	 Pred 1
Aktual 0: 191703		 4915
Aktual 1: 5264		 14615

--- Metrik per Kelas ---
KELAS           | PRECISION  | RECALL     | F1-SCORE  
-------------------------------------------------------
0 (Not Churn)   | 97.33%     | 97.50%     | 97.41%
1 (Churn)       | 74.83%     | 73.52%     | 74.17%
-------------------------------------------------------


### Feature Importance

In [ ]:
import pandas as pd
import re

def extract_feature_importance_fixed(model_pipeline, dataset, numerical_cols_list, features_col="features"):
    # 1. Ambil Model dari Stage Terakhir
    model = model_pipeline.stages[-1]
    
    if not hasattr(model, 'featureImportances'):
        print(f"Model {type(model).__name__} tidak memiliki attribut 'featureImportances'.")
        return None

    # 2. Ambil Score
    importances = model.featureImportances.toArray()
    
    # 3. Ambil Metadata Nama Fitur (dari kolom features di data hasil transform)
    meta = dataset.schema[features_col].metadata
    feature_names = []
    
    if "ml_attr" in meta:
        attrs = meta["ml_attr"]["attrs"]
        all_attrs = []
        if "numeric" in attrs: all_attrs += attrs["numeric"]
        if "nominal" in attrs: all_attrs += attrs["nominal"]
        if "binary" in attrs: all_attrs += attrs["binary"]
        
        # Sort berdasarkan index
        all_attrs.sort(key=lambda x: x['idx'])
        feature_names = [x['name'] for x in all_attrs]
    else:
        feature_names = [f"feature_{i}" for i in range(len(importances))]

    # 4. FUNGSI PERBAIKAN NAMA
    # Jika bertemu "scaled_numerical_features_X", akan diganti dengan numerical_cols_list[X]
    clean_names = []
    for name in feature_names:
        # Cek pola "scaled_numerical_features_" diikuti angka
        if "scaled_numerical_features" in name:
            try:
                # Ambil angka di bagian paling akhir string (misal: ..._12 -> 12)
                # Split berdasarkan underscore
                parts = name.split('_')
                # Indeks biasanya elemen terakhir
                idx = int(parts[-1])
                
                # Ambil nama asli dari list numerical_cols
                if idx < len(numerical_cols_list):
                    real_name = numerical_cols_list[idx]
                    clean_names.append(real_name) # Masukkan nama asli
                else:
                    clean_names.append(name) # Fallback jika index out of bound
            except:
                clean_names.append(name)
        else:
            # Jika bukan fitur numerik (misal: age_group_vec...), biarkan saja
            clean_names.append(name)

    # 5. Buat DataFrame
    df_importance = pd.DataFrame({
        'Feature_Name': clean_names,
        'Importance_Score': importances
    })
    
    # 6. Sort
    df_importance = df_importance.sort_values(by='Importance_Score', ascending=False).reset_index(drop=True)
    
    return df_importance

# Pastikan numerical_cols terdefinisi
numerical_cols = [
    'total_transactions', 'total_plan_days', 'avg_amount_paid', 'count_auto_renew', 'count_cancel',
    'most_frequent_payment_method', 'days_since_last_activity', 'total_secs_last_30d', 'active_days_last_30d', 
    'lifetime_active_days'
]

# numerical_cols = [
#     'total_transactions', 'total_plan_days', 'total_amount_paid', 'avg_amount_paid', 'count_auto_renew', 'count_cancel',
#     'most_frequent_payment_method', 'days_since_last_activity', 'total_secs_last_30d', 'active_days_last_30d', 'activity_ratio_secs',
#     'lifetime_active_days', 'lifetime_unq_songs'
# ]

print("--- Feature Importance: GBT Classifier (FIXED) ---")
# Masukkan numerical_cols ke dalam fungsi
df_imp_gbt = extract_feature_importance_fixed(model_gbt, preds_gbt, numerical_cols)

if df_imp_gbt is not None:
    # Tampilkan semua fitur agar terlihat jelas
    print(df_imp_gbt.to_string())

# print("\n--- Feature Importance: Random Forest (FIXED) ---")
# df_imp_rf = extract_feature_importance_fixed(model_rf, preds_rf, numerical_cols)

# if df_imp_rf is not None:
#     print(df_imp_rf.head(10))

--- Feature Importance: GBT Classifier (FIXED) ---
                    Feature_Name  Importance_Score
0               count_auto_renew          0.412089
1                   count_cancel          0.124668
2                avg_amount_paid          0.099397
3            total_secs_last_30d          0.093833
4       days_since_last_activity          0.080224
5           active_days_last_30d          0.076294
6             total_transactions          0.052790
7   most_frequent_payment_method          0.022790
8                total_plan_days          0.021992
9           lifetime_active_days          0.013007
10          registered_via_vec_7          0.002877
11          registered_via_vec_4          0.000020
12          registered_via_vec_3          0.000017
13          registered_via_vec_9          0.000000
14         registered_via_vec_13          0.000000
